In [ ]:
# The month and year with the price purchases

most_sales = fact_orders.groupBy("month", "year").agg(sum(col("price")).cast("decimal(18, 2)").alias("total_price"))\
            .orderBy("year", "month")

In [ ]:
# Top 5 brand from the total price purchases

top_5_brand = fact_orders.join(dim_products, "product_key", "inner")\
            .groupBy("brand").agg(sum(col("price")).cast("decimal(18, 2)").alias("total_price"))

window_spec = Window.orderBy(col("total_price").desc())

top_5_brand = top_5_brand.withColumn("top_5", dense_rank().over(window_spec)).limit(5)

In [ ]:
# Users that bought more than 1 products

users_that_bought_more_than_1_product = fact_orders.join(dim_users, "user_key", "inner") \
            .join(dim_products, "product_key", "inner") \
            .withColumn("more_than_1", dense_rank().over(Window.partitionBy("user_id").orderBy("product_id"))) \
            .filter(col("more_than_1") > 1) \
            .select("user_id").distinct()

In [ ]:
# Users that viewed a product, added the product to a cart and purchased that same product

full_users = (
    fact_user_events
        .join(dim_events, "event_key")
        .join(dim_users, "user_key")
        .groupBy("user_id")
        .agg(collect_set("event_type").alias("events"))
        .filter(
            array_contains(col("events"), "view") &
            array_contains(col("events"), "cart") &
            array_contains(col("events"), "purchase")
        )
)